### Automated Text Annotation with Ollama

This Jupyter Notebook demonstrates how to perform **automated text annotation** using **Ollama**.

We will:
1. Load and preprocess textual data
2. Use **Phi4** to classify and annotate text
3. Store and inspect the classified data

This approach enables efficient annotation at scale, reducing manual effort.

In [1]:
# Install required packages (uncomment if needed)
# !pip install pandas ollama

In [2]:
# Import necessary libraries
import pandas as pd 
from ollama import Client
import re

### 1. Function to Instruct LLM to Annotate

In [3]:
def classify_comment_to_df(row, system_message, model, options=None):
    """
    Classify a single comment and return its classification.
    """
    # Access the text from the DataFrame row
    text = row["text"]

    # Clean the text
    text = re.sub(r'\s+', ' ', text).strip()

    # Construct the input messages
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": text},
    ]

    if not options:
        options = {'seed': 42, 'temperature': 0.0, 'max_tokens': 3}

    try:
        # Request classification from the model
        response = client.chat(model, messages, options=options)
        result = response['message']['content']
    except Exception as e:
        print(f"Error classifying text: {text}\nError: {e}")
        result = "Error"

    return result

### 2. Setting up connection with Ollama

We have to initialize the client through Ollama and specify the pretrained model we want to use. In our case, we are using phi4:14b, a relatively new open-source Large Language Model. 

In [4]:
client = Client()
MODEL = 'phi4:14b'

In [ ]:
get_available_models = lambda : [m['model'] for m in client.list()['models']]

if MODEL not in get_available_models():
  print(f"Model not found. Running `client.pull('{MODEL}')` to download it.")
  client.pull(MODEL)

# confirm
assert MODEL in get_available_models()

In [ ]:
  print(f"Model not found. Running `client.pull('{MODEL}')` to download it.")
  client.pull(MODEL)

# confirm
assert MODEL in get_available_models()

### 3. Constructing Prompt with Instructions for the LLM

In [6]:
instructions = """
You are well-trained research assistant with an annotation task.
For each comment in the sample, follow these instructions:

1. Carefully read the text of the comment, paying close attention to details.
2. Classify the comment as either containing indications of the presence of a mental health disorder (1) or not containing indications of the presence of a mental health disorder (0).
"""

categories = ["Disorder", "No Disorder"]

defintions = """
Comments should be coded as DISORDER when they include any indications that could refer to a mental health disorder or its symptoms. 
Comments should be coded as NO DISORDER when they do not contain any indications that could refer to a mental health disorder or its symptoms. 
"""

In [7]:
prompt = f"Classify the following text into one of the given categories: {categories}\n{defintions}\nOnly include the selected category in your response and no further text."
print(prompt)

Classify the following text into one of the given categories: ['Disorder', 'No Disorder']

Comments should be coded as DISORDER when they include any indications that could refer to a mental health disorder or its symptoms. 
Comments should be coded as NO DISORDER when they do not contain any indications that could refer to a mental health disorder or its symptoms. 

Only include the selected category in your response and no further text.


### **4. Load Dataset and Sampling**

We load a dataset of comments, keeping only relevant columns. We draw a random sample to demonstrate annotation

In [8]:
# Define file path (update this path based on your dataset location)
file_path = "/Users/deniseroth/Documents/Downloads/mental_health_sentiment.csv"

# Load dataset with selected columns
df = pd.read_csv(file_path)

sampled_df = (
    df.sample(n=100, random_state=42))

print("Dataset Loaded Successfully! Sample preview:")
print(sampled_df.head())

Dataset Loaded Successfully! Sample preview:
          id                                               text   sentiment  \
33758  44409  christinastokes is sh working for you for me i...      Normal   
10308  13708  I started my first job, 5 months ago, and I do...  Depression   
4347    4347  Hey you, live long and healthy always. Good lu...      Normal   
19922  29913  I thought I did well but I was severely depres...      Stress   
1570    1570                                             #NAME?      Normal   

       condition_binary  
33758                 0  
10308                 1  
4347                  0  
19922                 1  
1570                  0  


### 5. Annotate Text

In [9]:
sampled_df["text"] = sampled_df["text"].fillna("").astype(str)
sampled_df["label"] = sampled_df.apply(
    lambda row: classify_comment_to_df(row, system_message=prompt, model=MODEL),
    axis=1
)

print(sampled_df[['text', 'label']].head())

                                                    text        label
33758  christinastokes is sh working for you for me i...  NO DISORDER
10308  I started my first job, 5 months ago, and I do...     Disorder
4347   Hey you, live long and healthy always. Good lu...  NO DISORDER
19922  I thought I did well but I was severely depres...     DISORDER
1570                                              #NAME?  NO DISORDER
